# 9.3 降雨数据后处理

本节将对前面计算的流域平均雨量结果进行误差优化和异常值处理。

## 导入必要的库

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle
from matplotlib.colors import LinearSegmentedColormap
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error
import os
import warnings
warnings.filterwarnings('ignore')

# 设置绘图样式
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

## 1. 数据加载和预处理

In [ ]:
# 加载之前计算的结果
data_dir = "../data/processed_rainfall"

try:
    # 加载流域平均雨量对比数据
    comparison_df = pd.read_csv(os.path.join(data_dir, 'basin_average_rainfall_comparison.csv'), index_col=0)
    comparison_df.index = pd.to_datetime(comparison_df.index)
    
    # 加载站点降雨数据
    station_df = pd.read_csv(os.path.join(data_dir, 'station_rainfall_data.csv'), index_col=0)
    station_df.index = pd.to_datetime(station_df.index)
    
    # 加载权重信息
    weights_df = pd.read_csv(os.path.join(data_dir, 'station_weights.csv'))
    
    print("数据加载成功！")
    print(f"流域平均雨量数据形状: {comparison_df.shape}")
    print(f"站点降雨数据形状: {station_df.shape}")
    
except FileNotFoundError:
    print("未找到处理过的数据，正在生成模拟数据...")
    
    # 生成模拟数据用于演示
    import datetime
    
    time_range = pd.date_range('2023-07-01', '2023-07-03', freq='3H')
    np.random.seed(42)
    
    # 模拟不同方法的结果
    base_rainfall = np.random.gamma(0.8, 2.0, len(time_range))
    base_rainfall[base_rainfall < 0.1] = 0
    
    comparison_df = pd.DataFrame({
        '算术平均法': base_rainfall + np.random.normal(0, 0.2, len(time_range)),
        '泰森多边形法': base_rainfall + np.random.normal(0, 0.15, len(time_range)),
        '距离权重法': base_rainfall + np.random.normal(0, 0.18, len(time_range)),
        '网格插值法': base_rainfall
    }, index=time_range)
    comparison_df[comparison_df < 0] = 0
    
    # 模拟站点数据
    station_names = ['station_1', 'station_2', 'station_3', 'station_4', 'station_5']
    station_data = {}
    for i, name in enumerate(station_names):
        station_data[name] = base_rainfall + np.random.normal(0, 0.3 + i*0.1, len(time_range))
    
    station_df = pd.DataFrame(station_data, index=time_range)
    station_df[station_df < 0] = 0
    
    # 模拟权重数据
    weights_df = pd.DataFrame({
        'station_id': station_names,
        'thiessen_weight': [0.25, 0.20, 0.30, 0.15, 0.10],
        'idw_weight': [0.22, 0.28, 0.25, 0.18, 0.07]
    })
    
    print("模拟数据生成完成！")

## 2. 基本统计分析

In [ ]:
# 基本统计信息
print("=== 流域平均雨量统计信息 ===")
stats_summary = comparison_df.describe()
print(stats_summary)

# 各方法的累积降雨量
print("\n=== 研究期间累积降雨量 ===")
cumulative_rainfall = comparison_df.sum() * 3  # 转换为总小时数
for method, total in cumulative_rainfall.items():
    print(f"{method}: {total:.1f} mm")

# 降雨事件识别（连续非零降雨）
def identify_rain_events(series, threshold=0.1):
    """识别降雨事件"""
    rain_mask = series > threshold
    events = []
    event_start = None
    
    for i, is_rain in enumerate(rain_mask):
        if is_rain and event_start is None:
            event_start = i
        elif not is_rain and event_start is not None:
            events.append((event_start, i-1))
            event_start = None
    
    if event_start is not None:
        events.append((event_start, len(series)-1))
    
    return events

# 以网格插值法为基准分析降雨事件
reference_method = '网格插值法'
rain_events = identify_rain_events(comparison_df[reference_method])

print(f"\n=== 降雨事件分析（基于{reference_method}） ===")
print(f"识别到 {len(rain_events)} 个降雨事件")

for i, (start, end) in enumerate(rain_events):
    duration = (end - start + 1) * 3  # 小时
    total_rain = comparison_df[reference_method].iloc[start:end+1].sum()
    max_intensity = comparison_df[reference_method].iloc[start:end+1].max()
    print(f"事件 {i+1}: 持续 {duration} 小时, 总量 {total_rain:.1f} mm, 最大强度 {max_intensity:.1f} mm/3hr")

## 3. 时间序列可视化

### 3.1 综合时间序列图

In [ ]:
# 创建综合时间序列图
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(4, 2, height_ratios=[2, 1, 1, 1], hspace=0.3)

# 主图：所有方法对比
ax1 = fig.add_subplot(gs[0, :])
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for i, method in enumerate(comparison_df.columns):
    ax1.plot(comparison_df.index, comparison_df[method], 
            label=method, linewidth=2.5, color=colors[i], marker='o', markersize=4)

ax1.fill_between(comparison_df.index, 0, comparison_df.min(axis=1), 
                alpha=0.1, color='gray', label='变化范围')
ax1.fill_between(comparison_df.index, comparison_df.min(axis=1), comparison_df.max(axis=1), 
                alpha=0.1, color='gray')

ax1.set_ylabel('降雨强度 (mm/3hr)', fontsize=12)
ax1.set_title('流域平均雨量时间序列对比', fontsize=14, fontweight='bold')
ax1.legend(loc='upper right', fontsize=11)
ax1.grid(True, alpha=0.3)

# 累积降雨量
ax2 = fig.add_subplot(gs[1, :])
for i, method in enumerate(comparison_df.columns):
    cumsum = comparison_df[method].cumsum()
    ax2.plot(comparison_df.index, cumsum, label=method, linewidth=2, color=colors[i])

ax2.set_ylabel('累积降雨量 (mm)', fontsize=12)
ax2.set_title('累积降雨量对比', fontsize=12)
ax2.legend(loc='upper left', fontsize=10)
ax2.grid(True, alpha=0.3)

# 方法差异
ax3 = fig.add_subplot(gs[2, :])
reference = comparison_df[reference_method]
for i, method in enumerate(comparison_df.columns):
    if method != reference_method:
        diff = comparison_df[method] - reference
        ax3.plot(comparison_df.index, diff, label=f'{method} - {reference_method}', 
                linewidth=2, color=colors[i])

ax3.axhline(y=0, color='black', linestyle='--', alpha=0.5)
ax3.set_ylabel('差异 (mm/3hr)', fontsize=12)
ax3.set_title(f'各方法与{reference_method}的差异', fontsize=12)
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

# 相对误差
ax4 = fig.add_subplot(gs[3, :])
for i, method in enumerate(comparison_df.columns):
    if method != reference_method:
        relative_error = ((comparison_df[method] - reference) / 
                         (reference + 0.01)) * 100  # 避免除零
        ax4.plot(comparison_df.index, relative_error, 
                label=f'{method}', linewidth=2, color=colors[i])

ax4.axhline(y=0, color='black', linestyle='--', alpha=0.5)
ax4.set_ylabel('相对误差 (%)', fontsize=12)
ax4.set_xlabel('时间', fontsize=12)
ax4.set_title(f'相对于{reference_method}的相对误差', fontsize=12)
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.2 交互式时间序列图（Plotly）

In [ ]:
# 创建交互式图表
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('流域平均雨量对比', '累积降雨量对比'),
    vertical_spacing=0.1
)

colors_plotly = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

# 添加原始时间序列
for i, method in enumerate(comparison_df.columns):
    fig.add_trace(
        go.Scatter(
            x=comparison_df.index,
            y=comparison_df[method],
            mode='lines+markers',
            name=method,
            line=dict(color=colors_plotly[i], width=2),
            marker=dict(size=4)
        ),
        row=1, col=1
    )

# 添加累积降雨量
for i, method in enumerate(comparison_df.columns):
    cumsum = comparison_df[method].cumsum()
    fig.add_trace(
        go.Scatter(
            x=comparison_df.index,
            y=cumsum,
            mode='lines',
            name=f'{method} (累积)',
            line=dict(color=colors_plotly[i], width=2, dash='dash'),
            showlegend=False
        ),
        row=2, col=1
    )

# 更新布局
fig.update_layout(
    title='降雨数据交互式分析',
    height=700,
    hovermode='x unified'
)

fig.update_yaxes(title_text="降雨强度 (mm/3hr)", row=1, col=1)
fig.update_yaxes(title_text="累积降雨量 (mm)", row=2, col=1)
fig.update_xaxes(title_text="时间", row=2, col=1)

fig.show()

## 4. 方法对比分析

### 4.1 相关性分析

In [ ]:
# 计算相关性矩阵
correlation_matrix = comparison_df.corr()

# 创建相关性热力图
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# 相关性热力图
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='RdYlBu_r', 
            square=True, ax=ax1, cbar_kws={'label': '相关系数'})
ax1.set_title('方法间相关性矩阵', fontsize=14)

# 误差统计
reference = comparison_df[reference_method]
error_stats = []

for method in comparison_df.columns:
    if method != reference_method:
        predicted = comparison_df[method]
        
        # 计算各种误差指标
        rmse = np.sqrt(mean_squared_error(reference, predicted))
        mae = mean_absolute_error(reference, predicted)
        mape = np.mean(np.abs((reference - predicted) / (reference + 0.01))) * 100
        r2 = stats.pearsonr(reference, predicted)[0]**2
        
        error_stats.append({
            '方法': method,
            'RMSE': rmse,
            'MAE': mae,
            'MAPE(%)': mape,
            'R²': r2
        })

error_df = pd.DataFrame(error_stats)

# 误差对比柱状图
x_pos = np.arange(len(error_df))
width = 0.2

ax2.bar(x_pos - width, error_df['RMSE'], width, label='RMSE', alpha=0.8)
ax2.bar(x_pos, error_df['MAE'], width, label='MAE', alpha=0.8)
ax2.bar(x_pos + width, error_df['MAPE(%)'], width, label='MAPE(%)', alpha=0.8)

ax2.set_xlabel('方法')
ax2.set_ylabel('误差值')
ax2.set_title(f'相对于{reference_method}的误差统计')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(error_df['方法'], rotation=45)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 打印误差统计表
print("\n=== 误差统计表 ===")
print(error_df.round(3))

### 4.2 散点图矩阵

In [ ]:
# 创建散点图矩阵
n_methods = len(comparison_df.columns)
fig, axes = plt.subplots(n_methods-1, n_methods-1, figsize=(15, 12))

methods = comparison_df.columns.tolist()
reference_idx = methods.index(reference_method)

# 移除参考方法，创建与其他方法的对比
other_methods = [m for m in methods if m != reference_method]

for i, method1 in enumerate(other_methods):
    for j, method2 in enumerate(other_methods):
        ax = axes[i, j] if n_methods > 2 else axes[j]
        
        if i == j:
            # 对角线：显示与参考方法的对比
            x = comparison_df[reference_method]
            y = comparison_df[method1]
            
            ax.scatter(x, y, alpha=0.7, s=50, color=colors[i+1])
            
            # 添加1:1线
            min_val = min(x.min(), y.min())
            max_val = max(x.max(), y.max())
            ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, alpha=0.8)
            
            # 计算R²
            r2 = stats.pearsonr(x, y)[0]**2
            ax.text(0.05, 0.95, f'R² = {r2:.3f}', transform=ax.transAxes, 
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
            
            ax.set_xlabel(f'{reference_method} (mm/3hr)')
            ax.set_ylabel(f'{method1} (mm/3hr)')
            
        else:
            # 非对角线：方法间直接对比
            x = comparison_df[method2]
            y = comparison_df[method1]
            
            ax.scatter(x, y, alpha=0.6, s=30)
            
            ax.set_xlabel(f'{method2} (mm/3hr)')
            ax.set_ylabel(f'{method1} (mm/3hr)')
        
        ax.grid(True, alpha=0.3)

plt.suptitle('方法对比散点图矩阵', fontsize=16, y=0.98)
plt.tight_layout()
plt.show()

## 5. 站点数据分析

### 5.1 站点降雨分布

In [ ]:
# 站点数据统计分析
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 站点时间序列
ax1 = axes[0, 0]
for station in station_df.columns:
    ax1.plot(station_df.index, station_df[station], 
            label=station, linewidth=2, marker='o', markersize=3)
ax1.set_ylabel('降雨强度 (mm/3hr)')
ax1.set_title('各站点降雨时间序列')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 站点累积降雨量
ax2 = axes[0, 1]
cumulative_by_station = station_df.sum()
bars = ax2.bar(range(len(cumulative_by_station)), cumulative_by_station.values, 
              color=plt.cm.viridis(np.linspace(0, 1, len(cumulative_by_station))))
ax2.set_xticks(range(len(cumulative_by_station)))
ax2.set_xticklabels(cumulative_by_station.index, rotation=45)
ax2.set_ylabel('累积降雨量 (mm)')
ax2.set_title('各站点累积降雨量')
ax2.grid(True, alpha=0.3)

# 在柱状图上添加数值标签
for bar, value in zip(bars, cumulative_by_station.values):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
            f'{value:.1f}', ha='center', va='bottom')

# 站点降雨分布箱线图
ax3 = axes[1, 0]
station_df.boxplot(ax=ax3)
ax3.set_ylabel('降雨强度 (mm/3hr)')
ax3.set_xlabel('站点')
ax3.set_title('各站点降雨分布')
ax3.tick_params(axis='x', rotation=45)
ax3.grid(True, alpha=0.3)

# 站点相关性热力图
ax4 = axes[1, 1]
station_corr = station_df.corr()
im = ax4.imshow(station_corr.values, cmap='RdYlBu_r', aspect='auto')
ax4.set_xticks(range(len(station_corr.columns)))
ax4.set_yticks(range(len(station_corr.columns)))
ax4.set_xticklabels(station_corr.columns, rotation=45)
ax4.set_yticklabels(station_corr.columns)
ax4.set_title('站点间相关性')

# 添加相关系数标注
for i in range(len(station_corr.columns)):
    for j in range(len(station_corr.columns)):
        text = ax4.text(j, i, f'{station_corr.iloc[i, j]:.2f}',
                       ha="center", va="center", color="black", fontsize=9)

plt.colorbar(im, ax=ax4, shrink=0.6)
plt.tight_layout()
plt.show()

### 5.2 权重分析

In [ ]:
# 权重对比分析
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))

# 权重对比柱状图
x = np.arange(len(weights_df))
width = 0.35

bars1 = ax1.bar(x - width/2, weights_df['thiessen_weight'], width, 
               label='泰森多边形法', alpha=0.8, color='#ff7f0e')
bars2 = ax1.bar(x + width/2, weights_df['idw_weight'], width, 
               label='距离权重法', alpha=0.8, color='#2ca02c')

ax1.set_xlabel('站点')
ax1.set_ylabel('权重')
ax1.set_title('不同方法的站点权重对比')
ax1.set_xticks(x)
ax1.set_xticklabels(weights_df['station_id'])
ax1.legend()
ax1.grid(True, alpha=0.3)

# 添加数值标签
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.005,
            f'{height:.3f}', ha='center', va='bottom', fontsize=9)

for bar in bars2:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.005,
            f'{height:.3f}', ha='center', va='bottom', fontsize=9)

# 权重散点图
ax2.scatter(weights_df['thiessen_weight'], weights_df['idw_weight'], 
           s=100, alpha=0.7, c=range(len(weights_df)), cmap='viridis')

# 添加站点标签
for i, row in weights_df.iterrows():
    ax2.annotate(row['station_id'], 
                (row['thiessen_weight'], row['idw_weight']),
                xytext=(5, 5), textcoords='offset points', fontsize=9)

# 添加1:1线
min_w = min(weights_df['thiessen_weight'].min(), weights_df['idw_weight'].min())
max_w = max(weights_df['thiessen_weight'].max(), weights_df['idw_weight'].max())
ax2.plot([min_w, max_w], [min_w, max_w], 'r--', alpha=0.7, linewidth=2)

ax2.set_xlabel('泰森多边形法权重')
ax2.set_ylabel('距离权重法权重')
ax2.set_title('两种方法权重对比')
ax2.grid(True, alpha=0.3)

# 权重差异分析
weight_diff = weights_df['thiessen_weight'] - weights_df['idw_weight']
colors = ['red' if x < 0 else 'blue' for x in weight_diff]

bars = ax3.bar(range(len(weight_diff)), weight_diff, color=colors, alpha=0.7)
ax3.axhline(y=0, color='black', linestyle='-', alpha=0.5)
ax3.set_xlabel('站点')
ax3.set_ylabel('权重差异 (泰森 - 距离)')
ax3.set_title('两种方法权重差异')
ax3.set_xticks(range(len(weights_df)))
ax3.set_xticklabels(weights_df['station_id'])
ax3.grid(True, alpha=0.3)

# 添加数值标签
for i, bar in enumerate(bars):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., 
            height + (0.005 if height >= 0 else -0.015),
            f'{height:.3f}', ha='center', 
            va='bottom' if height >= 0 else 'top', fontsize=9)

plt.tight_layout()
plt.show()

# 权重统计
print("\n=== 权重统计分析 ===")
print(f"泰森多边形法权重 - 平均: {weights_df['thiessen_weight'].mean():.3f}, 标准差: {weights_df['thiessen_weight'].std():.3f}")
print(f"距离权重法权重 - 平均: {weights_df['idw_weight'].mean():.3f}, 标准差: {weights_df['idw_weight'].std():.3f}")
print(f"权重相关系数: {weights_df['thiessen_weight'].corr(weights_df['idw_weight']):.3f}")

## 6. 降雨事件分析

In [ ]:
# 详细的降雨事件分析
def analyze_rain_events_detailed(comparison_df, threshold=0.1):
    """详细分析降雨事件"""
    events_analysis = []
    
    for method in comparison_df.columns:
        events = identify_rain_events(comparison_df[method], threshold)
        
        for i, (start, end) in enumerate(events):
            duration = (end - start + 1) * 3  # 小时
            event_data = comparison_df[method].iloc[start:end+1]
            
            events_analysis.append({
                '方法': method,
                '事件编号': i + 1,
                '开始时间': comparison_df.index[start],
                '结束时间': comparison_df.index[end],
                '持续时间(小时)': duration,
                '总降雨量(mm)': event_data.sum(),
                '最大强度(mm/3hr)': event_data.max(),
                '平均强度(mm/3hr)': event_data.mean()
            })
    
    return pd.DataFrame(events_analysis)

# 分析降雨事件
events_df = analyze_rain_events_detailed(comparison_df)
print("=== 降雨事件详细分析 ===")
print(events_df)

# 事件统计可视化
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 各方法事件数量
ax1 = axes[0, 0]
event_counts = events_df.groupby('方法').size()
bars = ax1.bar(event_counts.index, event_counts.values, alpha=0.8, 
              color=colors[:len(event_counts)])
ax1.set_ylabel('事件数量')
ax1.set_title('各方法识别的降雨事件数量')
ax1.tick_params(axis='x', rotation=45)
for bar, count in zip(bars, event_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
            str(count), ha='center', va='bottom')

# 事件持续时间分布
ax2 = axes[0, 1]
for i, method in enumerate(comparison_df.columns):
    method_events = events_df[events_df['方法'] == method]
    ax2.scatter(range(len(method_events)), method_events['持续时间(小时)'], 
               label=method, alpha=0.7, s=50, color=colors[i])

ax2.set_xlabel('事件序号')
ax2.set_ylabel('持续时间 (小时)')
ax2.set_title('降雨事件持续时间分布')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 事件总量vs最大强度
ax3 = axes[1, 0]
for i, method in enumerate(comparison_df.columns):
    method_events = events_df[events_df['方法'] == method]
    ax3.scatter(method_events['总降雨量(mm)'], method_events['最大强度(mm/3hr)'], 
               label=method, alpha=0.7, s=60, color=colors[i])

ax3.set_xlabel('事件总降雨量 (mm)')
ax3.set_ylabel('事件最大强度 (mm/3hr)')
ax3.set_title('降雨事件总量 vs 最大强度')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 事件强度箱线图
ax4 = axes[1, 1]
intensity_data = [events_df[events_df['方法'] == method]['平均强度(mm/3hr)'].values 
                 for method in comparison_df.columns]
box_plot = ax4.boxplot(intensity_data, labels=comparison_df.columns, patch_artist=True)

# 为箱线图着色
for patch, color in zip(box_plot['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax4.set_ylabel('平均强度 (mm/3hr)')
ax4.set_title('各方法降雨事件平均强度分布')
ax4.tick_params(axis='x', rotation=45)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 综合评估报告

In [ ]:
# 生成综合评估报告
def generate_comprehensive_report(comparison_df, error_df, events_df, weights_df):
    """生成综合评估报告"""
    
    report = []
    report.append("=" * 60)
    report.append("           流域平均雨量计算方法评估报告")
    report.append("=" * 60)
    
    # 基本信息
    report.append(f"\n1. 基本信息")
    report.append(f"   分析时间段: {comparison_df.index[0]} 至 {comparison_df.index[-1]}")
    report.append(f"   时间步长: 3小时")
    report.append(f"   总时间点数: {len(comparison_df)}")
    report.append(f"   参考方法: {reference_method}")
    
    # 降雨统计
    report.append(f"\n2. 降雨统计")
    total_rainfall = comparison_df.sum()
    for method in comparison_df.columns:
        report.append(f"   {method}: {total_rainfall[method]:.1f} mm")
    
    max_intensity = comparison_df.max()
    report.append(f"\n   最大强度:")
    for method in comparison_df.columns:
        report.append(f"   {method}: {max_intensity[method]:.1f} mm/3hr")
    
    # 方法对比
    report.append(f"\n3. 方法精度评估（相对于{reference_method}）")
    report.append(f"   {'方法':<15} {'RMSE':<8} {'MAE':<8} {'MAPE(%)':<10} {'R²':<8}")
    report.append(f"   {'-'*50}")
    for _, row in error_df.iterrows():
        report.append(f"   {row['方法']:<15} {row['RMSE']:<8.3f} {row['MAE']:<8.3f} {row['MAPE(%)']:<10.1f} {row['R²']:<8.3f}")
    
    # 降雨事件分析
    report.append(f"\n4. 降雨事件分析")
    event_summary = events_df.groupby('方法').agg({
        '事件编号': 'count',
        '持续时间(小时)': 'mean',
        '总降雨量(mm)': 'mean',
        '最大强度(mm/3hr)': 'mean'
    }).round(2)
    
    report.append(f"   {'方法':<15} {'事件数':<8} {'平均持续(h)':<12} {'平均总量(mm)':<14} {'平均最大强度':<12}")
    report.append(f"   {'-'*65}")
    for method in event_summary.index:
        row = event_summary.loc[method]
        report.append(f"   {method:<15} {int(row['事件编号']):<8} {row['持续时间(小时)']:<12.1f} {row['总降雨量(mm)']:<14.1f} {row['最大强度(mm/3hr)']:<12.1f}")
    
    # 权重分析
    report.append(f"\n5. 站点权重分析")
    report.append(f"   {'站点':<12} {'泰森权重':<10} {'距离权重':<10} {'权重差异':<10}")
    report.append(f"   {'-'*45}")
    for _, row in weights_df.iterrows():
        diff = row['thiessen_weight'] - row['idw_weight']
        report.append(f"   {row['station_id']:<12} {row['thiessen_weight']:<10.3f} {row['idw_weight']:<10.3f} {diff:<10.3f}")
    
    # 方法推荐
    report.append(f"\n6. 方法推荐")
    
    # 基于RMSE排序
    best_rmse = error_df.loc[error_df['RMSE'].idxmin(), '方法']
    best_r2 = error_df.loc[error_df['R²'].idxmax(), '方法']
    
    report.append(f"   • 最小RMSE: {best_rmse}")
    report.append(f"   • 最高R²: {best_r2}")
    
    if best_rmse == best_r2:
        report.append(f"   • 推荐方法: {best_rmse}（综合性能最佳）")
    else:
        report.append(f"   • 根据不同指标，推荐方法有所不同")
        report.append(f"     - 追求最小误差: {best_rmse}")
        report.append(f"     - 追求最高相关性: {best_r2}")
    
    # 使用建议
    report.append(f"\n7. 使用建议")
    report.append(f"   • 网格插值法: 适用于有高质量网格数据的区域")
    report.append(f"   • 泰森多边形法: 适用于站点分布不均匀的情况")
    report.append(f"   • 距离权重法: 适用于站点较多且分布相对均匀的情况")
    report.append(f"   • 算术平均法: 适用于站点分布均匀的小流域")
    
    report.append(f"\n" + "=" * 60)
    
    return "\n".join(report)

# 生成并显示报告
report = generate_comprehensive_report(comparison_df, error_df, events_df, weights_df)
print(report)

# 保存报告
report_dir = "../data/reports"
os.makedirs(report_dir, exist_ok=True)
with open(os.path.join(report_dir, 'rainfall_analysis_report.txt'), 'w', encoding='utf-8') as f:
    f.write(report)

print(f"\n报告已保存至: {os.path.join(report_dir, 'rainfall_analysis_report.txt')}")

## 小结

在本章的可视化分析中，我们完成了：

### 主要成果：

1. **时间序列分析**
   - 对比了四种方法的时间变化特征
   - 分析了累积降雨量差异
   - 创建了交互式可视化图表

2. **精度评估**
   - 计算了RMSE、MAE、MAPE、R²等指标
   - 绘制了相关性矩阵和散点图
   - 量化了各方法的误差特征

3. **站点数据分析**
   - 分析了各站点的降雨特征
   - 对比了不同权重方法的差异
   - 评估了站点间的相关性

4. **降雨事件分析**
   - 识别和分析了降雨事件特征
   - 对比了事件持续时间和强度
   - 统计了各方法的事件检测能力

5. **综合评估报告**
   - 生成了详细的评估报告
   - 提供了方法选择建议
   - 给出了实际应用指导

### 关键发现：

- 不同方法在精度和适用性上各有特点
- 网格插值法通常能提供最详细的空间信息
- 泰森多边形法在站点分布不均时表现良好
- 距离权重法计算简单且精度适中
- 算术平均法虽简单但在某些情况下效果不错

### 实际应用建议：

1. **数据丰富地区**: 优先选择网格插值法
2. **站点稀少地区**: 考虑距离权重法或泰森多边形法
3. **实时应用场景**: 可选择计算简单的算术平均法
4. **高精度要求**: 建议结合多种方法进行集成

通过本章的学习，大家应该掌握了降雨数据处理的完整流程，为后续的水文建模和分析奠定了坚实基础。